# Phase 7: Full Evaluation
## FreshCart AI — 4-System Comparative Evaluation

## 1. Setup & Data Loading

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os
from tensorflow import keras
from google.colab import drive

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/freshcart/data/processed'
MODELS_DIR = '/content/drive/MyDrive/freshcart/saved_models'
PRECOMP_DIR = '/content/drive/MyDrive/freshcart/data/precomputed'

# 1. Data
test_data = np.load(f'{DATA_DIR}/X_test.npz')
X_test = test_data['sequences']

with open(f'{DATA_DIR}/vocab.json', 'r') as f:
    vocab = json.load(f)

# 2. Models
lstm_model = keras.models.load_model(f'{MODELS_DIR}/lstm_model.h5')
lstm_time_model = keras.models.load_model(f'{MODELS_DIR}/lstm_time_model.keras')

# 3. Precomputed tables
with open(f'{PRECOMP_DIR}/global_top20.json', 'r') as f:
    global_top20 = json.load(f)
with open(f'{PRECOMP_DIR}/hourly_top10.json', 'r') as f:
    hourly_top10 = json.load(f)
with open(f'{PRECOMP_DIR}/dow_top10.json', 'r') as f:
    dow_top10 = json.load(f)
with open(f'{PRECOMP_DIR}/aisle_top10.json', 'r') as f:
    aisle_top10 = json.load(f)

# 4. Baseline results
try:
    baseline_results = pd.read_csv(f'{DATA_DIR}/baseline_metrics.csv')
except FileNotFoundError:
    baseline_results = None

print("Data, Models, and Tables loaded successfully.")
print(f"X_test shape: {X_test.shape}")

## 2. Dynamic Dataset Truncation for Full System

In [ ]:
def create_simulated_test_set(X_test_arr, random_seed=42):
    """
    Takes the array of test users and dynamically truncates their history
    to simulate the 3-tier cold start distribution:
    - 20% Tier 1 (0 items)
    - 30% Tier 2 (1 or 2 items randomly)
    - 50% Tier 3 (unmodified)
    Leaves the target label untouched.
    """
    np.random.seed(random_seed)
    X_sim = X_test_arr.copy()
    n_users = len(X_sim)
    
    # Create indices and shuffle to assign tiers
    indices = np.arange(n_users)
    np.random.shuffle(indices)
    
    t1_end = int(0.2 * n_users)
    t2_end = t1_end + int(0.3 * n_users)
    
    tier1_idx = indices[:t1_end]
    tier2_idx = indices[t1_end:t2_end]
    tier3_idx = indices[t2_end:]
    
    # Tier 1: length = 0 -> All context sequence elements to 0
    X_sim[tier1_idx, :-1] = 0
    
    # Tier 2: length = 1 or 2
    for idx in tier2_idx:
        keep_len = np.random.choice([1, 2])
        context = X_sim[idx, :-1]
        non_zero = context[context != 0]
        if len(non_zero) > keep_len:
            new_context = np.zeros_like(context)
            new_context[-keep_len:] = non_zero[-keep_len:]
            X_sim[idx, :-1] = new_context
            
    # Tier 3: Unmodified
    
    # Verification Check
    t1_count = 0
    t2_count = 0
    t3_count = 0
    for seq in X_sim[:, :-1]:
        non_zero = len(seq[seq != 0])
        if non_zero == 0:
            t1_count += 1
        elif non_zero in [1, 2]:
            t2_count += 1
        else:
            t3_count += 1
            
    print(f"{t1_count} Tier 1 sequences (expected {int(0.2*n_users)})")
    print(f"{t2_count} Tier 2 sequences (expected {int(0.3*n_users)})")
    print(f"{t3_count} Tier 3 sequences (expected {int(0.5*n_users)})")
    
    return X_sim

X_test_simulated = create_simulated_test_set(X_test)

## 3. Full System Router Logic

In [ ]:
def get_user_tier(order_count):
    if order_count == 0:
        return 1
    elif order_count in [1, 2]:
        return 2
    else:
        return 3

def get_recommendations(user_sequence, time_features):
    """
    user_sequence: 1D array of length 99 (context window, excluding target)
    time_features: tuple (hour, dow, days_gap)
    """
    non_zero = user_sequence[user_sequence != 0]
    order_count = len(non_zero)
    tier = get_user_tier(order_count)
    
    rec_list = []
    
    if tier == 1:
        # Global Top 20 -> return top 5
        rec_list = global_top20[:5]
        
    elif tier == 2:
        # Time-Aware Routing
        hour, dow, _ = time_features
        hour_str = str(hour)
        dow_str = str(dow)
        
        from_hour = hourly_top10.get(hour_str, [])
        from_dow = dow_top10.get(dow_str, [])
        
        # Combine and deduplicate
        combined = []
        seen = set()
        # Interleave to prioritize both equally
        for a, b in zip(from_hour + [None]*10, from_dow + [None]*10):
            if a is not None and a not in seen:
                combined.append(a)
                seen.add(a)
            if b is not None and b not in seen:
                combined.append(b)
                seen.add(b)
                
        rec_list = combined[:5]
        if len(rec_list) < 5:
            for item in global_top20:
                if item not in seen:
                    rec_list.append(item)
                    seen.add(item)
                if len(rec_list) == 5:
                    break
                    
    elif tier == 3:
        # Time-Aware LSTM Prediction
        seq_in = user_sequence.reshape(1, -1).astype(np.int32)
        hour, dow, days_gap = time_features
        h_in = np.array([hour], dtype=np.int32)
        d_in = np.array([dow], dtype=np.int32)
        g_in = np.array([days_gap], dtype=np.float32).reshape(1, 1)
        
        preds = lstm_time_model.predict({'seq_input': seq_in, 'hour_input': h_in, 'dow_input': d_in, 'days_gap_input': g_in}, verbose=0)
        top5_idx = np.argsort(preds[0])[-5:][::-1]
        rec_list = [int(x) for x in top5_idx]
        
    return rec_list
